In [1]:
import json
from pathlib import Path
import pandas as pd

In [2]:
PROJECT_ROOT = Path("..")

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
PATIENTS_DIR = PROJECT_ROOT / "data" / "synthetic_patients"

PATIENTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Trials folder:", TRIALS_DIR.resolve())
print("Synthetic patients folder:", PATIENTS_DIR.resolve())

Project root: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant
Trials folder: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\trials
Synthetic patients folder: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients


**Verifying the 5 selected trials**

In [3]:
five_trials_path = TRIALS_DIR / "five_proposed_trials.csv"

five_trials = pd.read_csv(five_trials_path)

print("Number of proposed trials:", len(five_trials))

five_trials[
    [
        "NCT Number",
        "Title",
        "Status as downloaded",
        "Minimum Age",
        "Maximum Age"
    ]
]

Number of proposed trials: 5


,NCT Number,Title,Status as downloaded,Minimum Age,Maximum Age
0,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,7 Years,22 Years
1,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,RECRUITING,1 Year,30 Years
2,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,2 Years,25 Years
3,NCT06235125,Safety and Feasibility of Intraoperative Visua...,RECRUITING,6 Years,17 Years
4,NCT02013336,Phase 1 Dose-escalating Study of MM-398 (Irino...,RECRUITING,12 Months,20 Years


## Defining Synthetic Patient Schema
Each synthetic patient will follow a consistent structure so that trial eligibility rules can be evaluated programmatically.

In [8]:
patient_template = {
    "patient_id": None,
    "age_years": None,
    "sex": None,
    
    "diagnosis": {
        "cancer_type": "osteosarcoma",
        "histologic_confirmation": None,
        "grade": None,
        "disease_status": None,
        "disease_extent": None,
        "metastatic_sites": [],
        "measurable_disease": None
    },
    
    "anthropometrics": {
        "weight_kg": None,
        "body_surface_area_m2": None
    },
    
    "performance_status": {
        "scale": None,
        "score": None
    },
    
    "prior_treatment": {
        "methotrexate": None,
        "doxorubicin": None,
        "cisplatin": None,
        "ifosfamide": None,
        "radiation": None,
        "major_surgery": None,
        "cellular_therapy": None,
        "investigational_agent": None
    },

    "current_treatment": {
    "map_chemotherapy": None,
    "methotrexate": None,
    "methotrexate_dose_g_m2": None,
    "doxorubicin": None,
    "cisplatin": None,
    "ifosfamide": None,
    "chemotherapy_cycle": None,
    "second_cycle_started": None
   },
    
    "treatment_timing_days": {
        "last_chemotherapy": None,
        "last_growth_factor": None,
        "last_radiation": None,
        "last_major_surgery": None,
        "last_cellular_therapy": None,
        "last_investigational_agent": None
    },
    
    "laboratory_values": {
        "anc_per_ul": None,
        "platelets_per_ul": None,
        "hemoglobin_g_dl": None,
        "bilirubin_uln_ratio": None,
        "alt_u_l": None,
        "alt_uln_ratio": None,
        "ast_uln_ratio": None,
        "alkaline_phosphatase_uln_ratio": None,
        "gfr_ml_min_1_73m2": None
    },
    
    "cardiac": {
        "lvef_percent": None,
        "qtc_ms": None
    },
    
    "pulmonary": {
        "oxygen_saturation_percent": None,
        "oxygen_requirement": None,
        "dyspnea_at_rest": None,
        "exercise_intolerance": None
    },
    
    "other_clinical_facts": {
        "active_infection": None,
        "pregnant": None,
        "breastfeeding": None,
        "life_expectancy_months": None,
        "blood_pressure_controlled": None,
        "prior_tegavivint": None,
        "prior_sulfatinib": None,
        "prior_vegf_vegfr_therapies_count": None
    },
    
    "surgical_imaging": {
        "pulmonary_metastatic_disease_suspected": None,
        "largest_pulmonary_nodule_mm": None,
        "pulmonary_surgery_recommended": None
    },
    
    "missing_information": []
}

In [9]:
patient_template

{'patient_id': None,
 'age_years': None,
 'sex': None,
 'diagnosis': {'cancer_type': 'osteosarcoma',
  'histologic_confirmation': None,
  'grade': None,
  'disease_status': None,
  'disease_extent': None,
  'metastatic_sites': [],
  'measurable_disease': None},
 'anthropometrics': {'weight_kg': None, 'body_surface_area_m2': None},
 'performance_status': {'scale': None, 'score': None},
 'prior_treatment': {'methotrexate': None,
  'doxorubicin': None,
  'cisplatin': None,
  'ifosfamide': None,
  'radiation': None,
  'major_surgery': None,
  'cellular_therapy': None,
  'investigational_agent': None},
 'current_treatment': {'map_chemotherapy': None,
  'methotrexate': None,
  'methotrexate_dose_g_m2': None,
  'doxorubicin': None,
  'cisplatin': None,
  'ifosfamide': None,
  'chemotherapy_cycle': None,
  'second_cycle_started': None},
 'treatment_timing_days': {'last_chemotherapy': None,
  'last_growth_factor': None,
  'last_radiation': None,
  'last_major_surgery': None,
  'last_cellular_th

## Creating Synthetic Patient P001
P001 is designed as a strong candidate for NCT07560826, the CATSINDO trial for newly diagnosed high-grade osteosarcoma.

In [11]:
import copy

P001 = copy.deepcopy(patient_template)

P001["patient_id"] = "P001"
P001["age_years"] = 15
P001["sex"] = "Male"

P001["diagnosis"]["histologic_confirmation"] = True
P001["diagnosis"]["grade"] = "high-grade"
P001["diagnosis"]["disease_status"] = "newly_diagnosed"
P001["diagnosis"]["disease_extent"] = "localized"
P001["diagnosis"]["metastatic_sites"] = []
P001["diagnosis"]["measurable_disease"] = True

P001["anthropometrics"]["weight_kg"] = 52
P001["anthropometrics"]["body_surface_area_m2"] = 1.55

P001["prior_treatment"]["methotrexate"] = False
P001["prior_treatment"]["doxorubicin"] = False
P001["prior_treatment"]["cisplatin"] = False

P001["current_treatment"]["map_chemotherapy"] = True
P001["current_treatment"]["methotrexate"] = True
P001["current_treatment"]["methotrexate_dose_g_m2"] = 12
P001["current_treatment"]["doxorubicin"] = True
P001["current_treatment"]["cisplatin"] = True
P001["current_treatment"]["ifosfamide"] = False
P001["current_treatment"]["chemotherapy_cycle"] = 1
P001["current_treatment"]["second_cycle_started"] = False
P001["prior_treatment"]["ifosfamide"] = False
P001["prior_treatment"]["radiation"] = False
P001["prior_treatment"]["major_surgery"] = False
P001["prior_treatment"]["cellular_therapy"] = False
P001["prior_treatment"]["investigational_agent"] = False

P001["laboratory_values"]["anc_per_ul"] = 1850
P001["laboratory_values"]["platelets_per_ul"] = 168000
P001["laboratory_values"]["hemoglobin_g_dl"] = 10.2
P001["laboratory_values"]["bilirubin_uln_ratio"] = 0.8
P001["laboratory_values"]["alt_u_l"] = 52
P001["laboratory_values"]["gfr_ml_min_1_73m2"] = 96

P001["cardiac"]["lvef_percent"] = 61

P001["other_clinical_facts"]["active_infection"] = False
P001["other_clinical_facts"]["pregnant"] = False
P001["other_clinical_facts"]["breastfeeding"] = False

P001["missing_information"] = []

In [12]:
print(json.dumps(P001, indent=2))

{
  "patient_id": "P001",
  "age_years": 15,
  "sex": "Male",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "newly_diagnosed",
    "disease_extent": "localized",
    "metastatic_sites": [],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": 52,
    "body_surface_area_m2": 1.55
  },
  "performance_status": {
    "scale": null,
    "score": null
  },
  "prior_treatment": {
    "methotrexate": false,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "radiation": false,
    "major_surgery": false,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": true,
    "methotrexate": true,
    "methotrexate_dose_g_m2": 12,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": false,
    "chemotherapy_cycle": 1,
    "second_cycle_started": false
  },
  "treatment_timing_days": {

## Creating Synthetic Patient P002
P002 represents a newly diagnosed high-grade osteosarcoma patient who matches the diagnosis, age, and treatment context for NCT07560826 but fails important hematologic and cardiac eligibility requirements.

In [13]:
P002 = copy.deepcopy(patient_template)

P002["patient_id"] = "P002"
P002["age_years"] = 16
P002["sex"] = "Female"

P002["diagnosis"]["histologic_confirmation"] = True
P002["diagnosis"]["grade"] = "high-grade"
P002["diagnosis"]["disease_status"] = "newly_diagnosed"
P002["diagnosis"]["disease_extent"] = "metastatic"
P002["diagnosis"]["metastatic_sites"] = ["lung"]
P002["diagnosis"]["measurable_disease"] = True

P002["anthropometrics"]["weight_kg"] = 50
P002["anthropometrics"]["body_surface_area_m2"] = 1.48

P002["prior_treatment"]["methotrexate"] = False
P002["prior_treatment"]["doxorubicin"] = False
P002["prior_treatment"]["cisplatin"] = False
P002["prior_treatment"]["ifosfamide"] = False
P002["prior_treatment"]["radiation"] = False
P002["prior_treatment"]["major_surgery"] = False
P002["prior_treatment"]["cellular_therapy"] = False
P002["prior_treatment"]["investigational_agent"] = False

P002["current_treatment"]["map_chemotherapy"] = True
P002["current_treatment"]["methotrexate"] = True
P002["current_treatment"]["methotrexate_dose_g_m2"] = 12
P002["current_treatment"]["doxorubicin"] = True
P002["current_treatment"]["cisplatin"] = True
P002["current_treatment"]["ifosfamide"] = False
P002["current_treatment"]["chemotherapy_cycle"] = 1
P002["current_treatment"]["second_cycle_started"] = False

P002["laboratory_values"]["anc_per_ul"] = 620
P002["laboratory_values"]["platelets_per_ul"] = 64000
P002["laboratory_values"]["hemoglobin_g_dl"] = 7.4
P002["laboratory_values"]["bilirubin_uln_ratio"] = 1.1
P002["laboratory_values"]["alt_u_l"] = 68
P002["laboratory_values"]["gfr_ml_min_1_73m2"] = 81

P002["cardiac"]["lvef_percent"] = 43

P002["other_clinical_facts"]["active_infection"] = False
P002["other_clinical_facts"]["pregnant"] = False
P002["other_clinical_facts"]["breastfeeding"] = False

P002["missing_information"] = []

In [14]:
print(json.dumps(P002, indent=2))

{
  "patient_id": "P002",
  "age_years": 16,
  "sex": "Female",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "newly_diagnosed",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": 50,
    "body_surface_area_m2": 1.48
  },
  "performance_status": {
    "scale": null,
    "score": null
  },
  "prior_treatment": {
    "methotrexate": false,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "radiation": false,
    "major_surgery": false,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": true,
    "methotrexate": true,
    "methotrexate_dose_g_m2": 12,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": false,
    "chemotherapy_cycle": 1,
    "second_cycle_started": false
  },
  "trea

## Creating Synthetic Patient P003
P003 represents a relapsed osteosarcoma patient designed as a strong candidate for NCT07144254.

In [15]:
P003 = copy.deepcopy(patient_template)

P003["patient_id"] = "P003"
P003["age_years"] = 17
P003["sex"] = "Male"

P003["diagnosis"]["histologic_confirmation"] = True
P003["diagnosis"]["grade"] = "high-grade"
P003["diagnosis"]["disease_status"] = "relapsed"
P003["diagnosis"]["disease_extent"] = "metastatic"
P003["diagnosis"]["metastatic_sites"] = ["lung"]
P003["diagnosis"]["measurable_disease"] = True

P003["performance_status"]["scale"] = "Karnofsky"
P003["performance_status"]["score"] = 80

P003["prior_treatment"]["methotrexate"] = True
P003["prior_treatment"]["doxorubicin"] = True
P003["prior_treatment"]["cisplatin"] = True
P003["prior_treatment"]["ifosfamide"] = False
P003["prior_treatment"]["radiation"] = False
P003["prior_treatment"]["major_surgery"] = True
P003["prior_treatment"]["cellular_therapy"] = False
P003["prior_treatment"]["investigational_agent"] = False

P003["current_treatment"]["map_chemotherapy"] = False
P003["current_treatment"]["methotrexate"] = False
P003["current_treatment"]["doxorubicin"] = False
P003["current_treatment"]["cisplatin"] = False
P003["current_treatment"]["ifosfamide"] = False
P003["current_treatment"]["chemotherapy_cycle"] = None
P003["current_treatment"]["second_cycle_started"] = None

P003["treatment_timing_days"]["last_chemotherapy"] = 28
P003["treatment_timing_days"]["last_growth_factor"] = 21
P003["treatment_timing_days"]["last_radiation"] = None
P003["treatment_timing_days"]["last_major_surgery"] = 35
P003["treatment_timing_days"]["last_cellular_therapy"] = None
P003["treatment_timing_days"]["last_investigational_agent"] = None

P003["laboratory_values"]["anc_per_ul"] = 1420
P003["laboratory_values"]["platelets_per_ul"] = 118000
P003["laboratory_values"]["bilirubin_uln_ratio"] = 0.9
P003["laboratory_values"]["alt_uln_ratio"] = 1.4
P003["laboratory_values"]["gfr_ml_min_1_73m2"] = 88

P003["cardiac"]["qtc_ms"] = 442

P003["pulmonary"]["oxygen_saturation_percent"] = 97
P003["pulmonary"]["oxygen_requirement"] = False
P003["pulmonary"]["dyspnea_at_rest"] = False
P003["pulmonary"]["exercise_intolerance"] = False

P003["other_clinical_facts"]["active_infection"] = False
P003["other_clinical_facts"]["prior_tegavivint"] = False

P003["missing_information"] = []

In [16]:
print(json.dumps(P003, indent=2))

{
  "patient_id": "P003",
  "age_years": 17,
  "sex": "Male",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "relapsed",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": null,
    "body_surface_area_m2": null
  },
  "performance_status": {
    "scale": "Karnofsky",
    "score": 80
  },
  "prior_treatment": {
    "methotrexate": true,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": false,
    "radiation": false,
    "major_surgery": true,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": false,
    "methotrexate": false,
    "methotrexate_dose_g_m2": null,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "chemotherapy_cycle": null,
    "second_cycle_started": null
  },
  "tr

## Creating Synthetic Patient P004
P004 represents a refractory osteosarcoma patient with intentionally missing information to test UNKNOWN eligibility states and follow-up question generation.

In [17]:
P004 = copy.deepcopy(patient_template)

P004["patient_id"] = "P004"
P004["age_years"] = 14
P004["sex"] = "Female"

P004["diagnosis"]["histologic_confirmation"] = True
P004["diagnosis"]["grade"] = "high-grade"
P004["diagnosis"]["disease_status"] = "refractory"
P004["diagnosis"]["disease_extent"] = "metastatic"
P004["diagnosis"]["metastatic_sites"] = ["lung"]
P004["diagnosis"]["measurable_disease"] = True

P004["performance_status"]["scale"] = "Lansky"
P004["performance_status"]["score"] = 70

P004["prior_treatment"]["methotrexate"] = True
P004["prior_treatment"]["doxorubicin"] = True
P004["prior_treatment"]["cisplatin"] = True
P004["prior_treatment"]["ifosfamide"] = None
P004["prior_treatment"]["radiation"] = False
P004["prior_treatment"]["major_surgery"] = True
P004["prior_treatment"]["cellular_therapy"] = False
P004["prior_treatment"]["investigational_agent"] = False

P004["current_treatment"]["map_chemotherapy"] = False
P004["current_treatment"]["methotrexate"] = False
P004["current_treatment"]["doxorubicin"] = False
P004["current_treatment"]["cisplatin"] = False
P004["current_treatment"]["ifosfamide"] = False
P004["current_treatment"]["chemotherapy_cycle"] = None
P004["current_treatment"]["second_cycle_started"] = None

P004["treatment_timing_days"]["last_chemotherapy"] = None
P004["treatment_timing_days"]["last_growth_factor"] = None
P004["treatment_timing_days"]["last_radiation"] = None
P004["treatment_timing_days"]["last_major_surgery"] = 28
P004["treatment_timing_days"]["last_cellular_therapy"] = None
P004["treatment_timing_days"]["last_investigational_agent"] = None

P004["laboratory_values"]["anc_per_ul"] = 1100
P004["laboratory_values"]["platelets_per_ul"] = 92000
P004["laboratory_values"]["bilirubin_uln_ratio"] = 1.0
P004["laboratory_values"]["alt_uln_ratio"] = None
P004["laboratory_values"]["gfr_ml_min_1_73m2"] = None

P004["cardiac"]["qtc_ms"] = None

P004["pulmonary"]["oxygen_saturation_percent"] = 96
P004["pulmonary"]["oxygen_requirement"] = False
P004["pulmonary"]["dyspnea_at_rest"] = False
P004["pulmonary"]["exercise_intolerance"] = False

P004["other_clinical_facts"]["active_infection"] = False
P004["other_clinical_facts"]["prior_tegavivint"] = False

P004["missing_information"] = [
    "ifosfamide treatment history",
    "date of last chemotherapy",
    "date of last growth factor treatment",
    "ALT result",
    "GFR or creatinine clearance",
    "QTc measurement"
]

In [18]:
print(json.dumps(P004, indent=2))

{
  "patient_id": "P004",
  "age_years": 14,
  "sex": "Female",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "refractory",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": null,
    "body_surface_area_m2": null
  },
  "performance_status": {
    "scale": "Lansky",
    "score": 70
  },
  "prior_treatment": {
    "methotrexate": true,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": null,
    "radiation": false,
    "major_surgery": true,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": false,
    "methotrexate": false,
    "methotrexate_dose_g_m2": null,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "chemotherapy_cycle": null,
    "second_cycle_started": null
  },
  "tr

## Creating Synthetic Patient P005
P005 represents a pediatric patient with relapsed/refractory high-grade osteosarcoma who is designed as a strong candidate for NCT05590572.

In [19]:
P005 = copy.deepcopy(patient_template)

P005["patient_id"] = "P005"
P005["age_years"] = 12
P005["sex"] = "Male"

P005["diagnosis"]["histologic_confirmation"] = True
P005["diagnosis"]["grade"] = "high-grade"
P005["diagnosis"]["disease_status"] = "refractory"
P005["diagnosis"]["disease_extent"] = "metastatic"
P005["diagnosis"]["metastatic_sites"] = ["lung"]
P005["diagnosis"]["measurable_disease"] = True

P005["performance_status"]["scale"] = "Lansky"
P005["performance_status"]["score"] = 80

P005["prior_treatment"]["methotrexate"] = True
P005["prior_treatment"]["doxorubicin"] = True
P005["prior_treatment"]["cisplatin"] = True
P005["prior_treatment"]["ifosfamide"] = True
P005["prior_treatment"]["radiation"] = False
P005["prior_treatment"]["major_surgery"] = True
P005["prior_treatment"]["cellular_therapy"] = False
P005["prior_treatment"]["investigational_agent"] = False

P005["current_treatment"]["map_chemotherapy"] = False
P005["current_treatment"]["methotrexate"] = False
P005["current_treatment"]["doxorubicin"] = False
P005["current_treatment"]["cisplatin"] = False
P005["current_treatment"]["ifosfamide"] = False
P005["current_treatment"]["chemotherapy_cycle"] = None
P005["current_treatment"]["second_cycle_started"] = None

P005["laboratory_values"]["anc_per_ul"] = 1700
P005["laboratory_values"]["platelets_per_ul"] = 126000
P005["laboratory_values"]["hemoglobin_g_dl"] = 9.6
P005["laboratory_values"]["bilirubin_uln_ratio"] = 0.7
P005["laboratory_values"]["alt_uln_ratio"] = 1.8
P005["laboratory_values"]["ast_uln_ratio"] = 1.6
P005["laboratory_values"]["alkaline_phosphatase_uln_ratio"] = 2.2
P005["laboratory_values"]["gfr_ml_min_1_73m2"] = 92

P005["cardiac"]["lvef_percent"] = 57

P005["other_clinical_facts"]["active_infection"] = False
P005["other_clinical_facts"]["life_expectancy_months"] = 9
P005["other_clinical_facts"]["blood_pressure_controlled"] = True
P005["other_clinical_facts"]["prior_sulfatinib"] = False
P005["other_clinical_facts"]["prior_vegf_vegfr_therapies_count"] = 1
P005["other_clinical_facts"]["pregnant"] = False
P005["other_clinical_facts"]["breastfeeding"] = False

P005["missing_information"] = []

In [20]:
print(json.dumps(P005, indent=2))

{
  "patient_id": "P005",
  "age_years": 12,
  "sex": "Male",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "refractory",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": null,
    "body_surface_area_m2": null
  },
  "performance_status": {
    "scale": "Lansky",
    "score": 80
  },
  "prior_treatment": {
    "methotrexate": true,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": true,
    "radiation": false,
    "major_surgery": true,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": false,
    "methotrexate": false,
    "methotrexate_dose_g_m2": null,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "chemotherapy_cycle": null,
    "second_cycle_started": null
  },
  "trea

## Creating Synthetic Patient P006
P006 represents a pediatric osteosarcoma patient with suspected pulmonary metastatic disease who is designed as a strong candidate for NCT06235125.

In [21]:
P006 = copy.deepcopy(patient_template)

P006["patient_id"] = "P006"
P006["age_years"] = 13
P006["sex"] = "Male"

P006["diagnosis"]["histologic_confirmation"] = True
P006["diagnosis"]["grade"] = "high-grade"
P006["diagnosis"]["disease_status"] = "metastatic"
P006["diagnosis"]["disease_extent"] = "metastatic"
P006["diagnosis"]["metastatic_sites"] = ["lung"]
P006["diagnosis"]["measurable_disease"] = True

P006["anthropometrics"]["weight_kg"] = 46
P006["anthropometrics"]["body_surface_area_m2"] = 1.42

P006["prior_treatment"]["methotrexate"] = True
P006["prior_treatment"]["doxorubicin"] = True
P006["prior_treatment"]["cisplatin"] = True
P006["prior_treatment"]["ifosfamide"] = False
P006["prior_treatment"]["radiation"] = False
P006["prior_treatment"]["major_surgery"] = False
P006["prior_treatment"]["cellular_therapy"] = False
P006["prior_treatment"]["investigational_agent"] = False

P006["current_treatment"]["map_chemotherapy"] = False
P006["current_treatment"]["methotrexate"] = False
P006["current_treatment"]["doxorubicin"] = False
P006["current_treatment"]["cisplatin"] = False
P006["current_treatment"]["ifosfamide"] = False
P006["current_treatment"]["chemotherapy_cycle"] = None
P006["current_treatment"]["second_cycle_started"] = None

P006["laboratory_values"]["bilirubin_uln_ratio"] = 0.8
P006["laboratory_values"]["alt_uln_ratio"] = 1.2
P006["laboratory_values"]["ast_uln_ratio"] = 1.0
P006["laboratory_values"]["alkaline_phosphatase_uln_ratio"] = 1.4
P006["laboratory_values"]["gfr_ml_min_1_73m2"] = 93

P006["other_clinical_facts"]["active_infection"] = False
P006["other_clinical_facts"]["pregnant"] = False
P006["other_clinical_facts"]["breastfeeding"] = False

P006["surgical_imaging"]["pulmonary_metastatic_disease_suspected"] = True
P006["surgical_imaging"]["largest_pulmonary_nodule_mm"] = 8
P006["surgical_imaging"]["pulmonary_surgery_recommended"] = True

P006["missing_information"] = []

In [22]:
print(json.dumps(P006, indent=2))

{
  "patient_id": "P006",
  "age_years": 13,
  "sex": "Male",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "metastatic",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": 46,
    "body_surface_area_m2": 1.42
  },
  "performance_status": {
    "scale": null,
    "score": null
  },
  "prior_treatment": {
    "methotrexate": true,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": false,
    "radiation": false,
    "major_surgery": false,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": false,
    "methotrexate": false,
    "methotrexate_dose_g_m2": null,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "chemotherapy_cycle": null,
    "second_cycle_started": null
  },
  "treatm

## Creating Synthetic Patient P007
P007 represents a pediatric patient with recurrent osteosarcoma and disease progression after prior therapy, designed as a strong candidate for NCT02013336.

In [23]:
P007 = copy.deepcopy(patient_template)

P007["patient_id"] = "P007"
P007["age_years"] = 10
P007["sex"] = "Female"

P007["diagnosis"]["histologic_confirmation"] = True
P007["diagnosis"]["grade"] = "high-grade"
P007["diagnosis"]["disease_status"] = "recurrent"
P007["diagnosis"]["disease_extent"] = "metastatic"
P007["diagnosis"]["metastatic_sites"] = ["lung"]
P007["diagnosis"]["measurable_disease"] = True

P007["performance_status"]["scale"] = "Lansky"
P007["performance_status"]["score"] = 80

P007["prior_treatment"]["methotrexate"] = True
P007["prior_treatment"]["doxorubicin"] = True
P007["prior_treatment"]["cisplatin"] = True
P007["prior_treatment"]["ifosfamide"] = False
P007["prior_treatment"]["radiation"] = False
P007["prior_treatment"]["major_surgery"] = True
P007["prior_treatment"]["cellular_therapy"] = False
P007["prior_treatment"]["investigational_agent"] = False

P007["current_treatment"]["map_chemotherapy"] = False
P007["current_treatment"]["methotrexate"] = False
P007["current_treatment"]["doxorubicin"] = False
P007["current_treatment"]["cisplatin"] = False
P007["current_treatment"]["ifosfamide"] = False
P007["current_treatment"]["chemotherapy_cycle"] = None
P007["current_treatment"]["second_cycle_started"] = None

P007["treatment_timing_days"]["last_chemotherapy"] = 35
P007["treatment_timing_days"]["last_growth_factor"] = 21
P007["treatment_timing_days"]["last_radiation"] = None
P007["treatment_timing_days"]["last_major_surgery"] = 42
P007["treatment_timing_days"]["last_cellular_therapy"] = None
P007["treatment_timing_days"]["last_investigational_agent"] = None

P007["laboratory_values"]["anc_per_ul"] = 1600
P007["laboratory_values"]["platelets_per_ul"] = 135000
P007["laboratory_values"]["hemoglobin_g_dl"] = 10.1
P007["laboratory_values"]["bilirubin_uln_ratio"] = 0.9
P007["laboratory_values"]["alt_uln_ratio"] = 1.3
P007["laboratory_values"]["gfr_ml_min_1_73m2"] = 90

P007["cardiac"]["lvef_percent"] = 58

P007["other_clinical_facts"]["active_infection"] = False
P007["other_clinical_facts"]["pregnant"] = False
P007["other_clinical_facts"]["breastfeeding"] = False

P007["missing_information"] = []

In [24]:
print(json.dumps(P007, indent=2))

{
  "patient_id": "P007",
  "age_years": 10,
  "sex": "Female",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "recurrent",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": null,
    "body_surface_area_m2": null
  },
  "performance_status": {
    "scale": "Lansky",
    "score": 80
  },
  "prior_treatment": {
    "methotrexate": true,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": false,
    "radiation": false,
    "major_surgery": true,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": false,
    "methotrexate": false,
    "methotrexate_dose_g_m2": null,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "chemotherapy_cycle": null,
    "second_cycle_started": null
  },
  "tr

## Creating Synthetic Patient P008
P008 is a cross-trial borderline osteosarcoma case designed to produce different MET, NOT_MET, and potentially UNKNOWN outcomes across the five selected trials.

In [25]:
P008 = copy.deepcopy(patient_template)

P008["patient_id"] = "P008"
P008["age_years"] = 19
P008["sex"] = "Male"

P008["diagnosis"]["histologic_confirmation"] = True
P008["diagnosis"]["grade"] = "high-grade"
P008["diagnosis"]["disease_status"] = "relapsed"
P008["diagnosis"]["disease_extent"] = "metastatic"
P008["diagnosis"]["metastatic_sites"] = ["lung"]
P008["diagnosis"]["measurable_disease"] = True

P008["anthropometrics"]["weight_kg"] = 61
P008["anthropometrics"]["body_surface_area_m2"] = 1.72

P008["performance_status"]["scale"] = "Karnofsky"
P008["performance_status"]["score"] = 70

P008["prior_treatment"]["methotrexate"] = True
P008["prior_treatment"]["doxorubicin"] = True
P008["prior_treatment"]["cisplatin"] = True
P008["prior_treatment"]["ifosfamide"] = True
P008["prior_treatment"]["radiation"] = False
P008["prior_treatment"]["major_surgery"] = True
P008["prior_treatment"]["cellular_therapy"] = False
P008["prior_treatment"]["investigational_agent"] = False

P008["current_treatment"]["map_chemotherapy"] = False
P008["current_treatment"]["methotrexate"] = False
P008["current_treatment"]["doxorubicin"] = False
P008["current_treatment"]["cisplatin"] = False
P008["current_treatment"]["ifosfamide"] = False
P008["current_treatment"]["chemotherapy_cycle"] = None
P008["current_treatment"]["second_cycle_started"] = None

P008["treatment_timing_days"]["last_chemotherapy"] = 10
P008["treatment_timing_days"]["last_growth_factor"] = None
P008["treatment_timing_days"]["last_radiation"] = None
P008["treatment_timing_days"]["last_major_surgery"] = 8
P008["treatment_timing_days"]["last_cellular_therapy"] = None
P008["treatment_timing_days"]["last_investigational_agent"] = None

P008["laboratory_values"]["anc_per_ul"] = 820
P008["laboratory_values"]["platelets_per_ul"] = 82000
P008["laboratory_values"]["hemoglobin_g_dl"] = 9.2
P008["laboratory_values"]["bilirubin_uln_ratio"] = 1.2
P008["laboratory_values"]["alt_uln_ratio"] = 3.8
P008["laboratory_values"]["gfr_ml_min_1_73m2"] = 74

P008["cardiac"]["lvef_percent"] = 52
P008["cardiac"]["qtc_ms"] = 476

P008["pulmonary"]["oxygen_saturation_percent"] = 96
P008["pulmonary"]["oxygen_requirement"] = False
P008["pulmonary"]["dyspnea_at_rest"] = False
P008["pulmonary"]["exercise_intolerance"] = False

P008["other_clinical_facts"]["active_infection"] = False
P008["other_clinical_facts"]["prior_tegavivint"] = False
P008["other_clinical_facts"]["prior_sulfatinib"] = False
P008["other_clinical_facts"]["prior_vegf_vegfr_therapies_count"] = 1

P008["surgical_imaging"]["pulmonary_metastatic_disease_suspected"] = True
P008["surgical_imaging"]["largest_pulmonary_nodule_mm"] = 9
P008["surgical_imaging"]["pulmonary_surgery_recommended"] = True

P008["missing_information"] = [
    "date of last growth factor treatment"
]

In [26]:
print(json.dumps(P008, indent=2))

{
  "patient_id": "P008",
  "age_years": 19,
  "sex": "Male",
  "diagnosis": {
    "cancer_type": "osteosarcoma",
    "histologic_confirmation": true,
    "grade": "high-grade",
    "disease_status": "relapsed",
    "disease_extent": "metastatic",
    "metastatic_sites": [
      "lung"
    ],
    "measurable_disease": true
  },
  "anthropometrics": {
    "weight_kg": 61,
    "body_surface_area_m2": 1.72
  },
  "performance_status": {
    "scale": "Karnofsky",
    "score": 70
  },
  "prior_treatment": {
    "methotrexate": true,
    "doxorubicin": true,
    "cisplatin": true,
    "ifosfamide": true,
    "radiation": false,
    "major_surgery": true,
    "cellular_therapy": false,
    "investigational_agent": false
  },
  "current_treatment": {
    "map_chemotherapy": false,
    "methotrexate": false,
    "methotrexate_dose_g_m2": null,
    "doxorubicin": false,
    "cisplatin": false,
    "ifosfamide": false,
    "chemotherapy_cycle": null,
    "second_cycle_started": null
  },
  "treat

## Combining Synthetic Patient Profiles
Combine all eight synthetic patient dictionaries into a single dataset for validation and export.

In [27]:
synthetic_patients = [
    P001,
    P002,
    P003,
    P004,
    P005,
    P006,
    P007,
    P008
]

print("Number of synthetic patients:", len(synthetic_patients))

Number of synthetic patients: 8


In [28]:
patient_ids = [patient["patient_id"] for patient in synthetic_patients]

print("Patient IDs:", patient_ids)
print("Unique patient IDs:", len(set(patient_ids)))

assert len(patient_ids) == 8
assert len(set(patient_ids)) == 8

print("Patient ID validation passed.")

Patient IDs: ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008']
Unique patient IDs: 8
Patient ID validation passed.


In [29]:
template_keys = set(patient_template.keys())

for patient in synthetic_patients:
    patient_keys = set(patient.keys())

    assert patient_keys == template_keys, (
        f"{patient['patient_id']} does not match the patient schema."
    )

print("All patients follow the same top-level schema.")

All patients follow the same top-level schema.


In [30]:
for patient in synthetic_patients:

    assert patient["patient_id"] is not None

    assert isinstance(patient["age_years"], (int, float))
    assert patient["age_years"] > 0

    assert patient["diagnosis"]["cancer_type"] == "osteosarcoma"

    assert isinstance(patient["diagnosis"]["metastatic_sites"], list)

    assert isinstance(patient["missing_information"], list)

print("Basic patient data validation passed.")

Basic patient data validation passed.


In [31]:
patient_summary = []

for patient in synthetic_patients:
    patient_summary.append({
        "Patient ID": patient["patient_id"],
        "Age": patient["age_years"],
        "Sex": patient["sex"],
        "Diagnosis": patient["diagnosis"]["cancer_type"],
        "Grade": patient["diagnosis"]["grade"],
        "Disease Status": patient["diagnosis"]["disease_status"],
        "Disease Extent": patient["diagnosis"]["disease_extent"],
        "Metastatic Sites": ", ".join(patient["diagnosis"]["metastatic_sites"]),
        "Measurable Disease": patient["diagnosis"]["measurable_disease"],
        "Missing Information Count": len(patient["missing_information"])
    })

patient_summary_df = pd.DataFrame(patient_summary)

patient_summary_df

,Patient ID,Age,Sex,Diagnosis,Grade,Disease Status,Disease Extent,Metastatic Sites,Measurable Disease,Missing Information Count
0,P001,15,Male,osteosarcoma,high-grade,newly_diagnosed,localized,,True,0
1,P002,16,Female,osteosarcoma,high-grade,newly_diagnosed,metastatic,lung,True,0
2,P003,17,Male,osteosarcoma,high-grade,relapsed,metastatic,lung,True,0
3,P004,14,Female,osteosarcoma,high-grade,refractory,metastatic,lung,True,6
4,P005,12,Male,osteosarcoma,high-grade,refractory,metastatic,lung,True,0
5,P006,13,Male,osteosarcoma,high-grade,metastatic,metastatic,lung,True,0
6,P007,10,Female,osteosarcoma,high-grade,recurrent,metastatic,lung,True,0
7,P008,19,Male,osteosarcoma,high-grade,relapsed,metastatic,lung,True,1


In [32]:
json_output_path = PATIENTS_DIR / "synthetic_patients.json"

with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(
        synthetic_patients,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved JSON to:")
print(json_output_path.resolve())

Saved JSON to:
D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients\synthetic_patients.json


In [33]:
csv_output_path = PATIENTS_DIR / "synthetic_patients_summary.csv"

patient_summary_df.to_csv(
    csv_output_path,
    index=False
)

print("Saved CSV summary to:")
print(csv_output_path.resolve())

Saved CSV summary to:
D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients\synthetic_patients_summary.csv


In [34]:
print("JSON exists:", json_output_path.exists())
print("CSV exists:", csv_output_path.exists())

print("\nFiles in synthetic patient directory:")

for file in PATIENTS_DIR.iterdir():
    print("-", file.name)

JSON exists: True
CSV exists: True

Files in synthetic patient directory:
- .gitkeep.txt
- synthetic_patients.json
- synthetic_patients_summary.csv


In [35]:
with open(json_output_path, "r", encoding="utf-8") as f:
    loaded_patients = json.load(f)

print("Patients loaded from JSON:", len(loaded_patients))

for patient in loaded_patients:
    print(
        patient["patient_id"],
        "-",
        patient["age_years"],
        "-",
        patient["diagnosis"]["disease_status"]
    )

Patients loaded from JSON: 8
P001 - 15 - newly_diagnosed
P002 - 16 - newly_diagnosed
P003 - 17 - relapsed
P004 - 14 - refractory
P005 - 12 - refractory
P006 - 13 - metastatic
P007 - 10 - recurrent
P008 - 19 - relapsed
